[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/03_%E7%B5%B1%E8%A8%88_3%E7%B4%9A/05_%E5%AE%9F%E9%A8%93%E3%81%A8%E6%9D%A1%E4%BB%B6%E4%BB%98%E3%81%8D%E7%A2%BA%E7%8E%87.ipynb)

> Colabで実行可。最初に「① セットアップ」セルを実行（Colabはデータを自動生成、ローカルは何もしない）。

# 統計3級-05. 実験・観察研究と確率（条件付き・独立）

**できるようになること**
- 観察研究と実験研究・処理群/対照群を説明できる
- 独立な試行の確率を計算できる
- 条件付き確率をクロス表から求められる

**前提**：統計3級01　/　**所要**：約25分　/　**レベル**：統計検定3級相当

In [ ]:
import numpy as np               # 数値計算
import pandas as pd              # 表データ
rng = np.random.default_rng(3)   # 乱数生成器（seedで結果を固定）

## 1. 観察研究 と 実験研究

| | 観察研究 | 実験研究 |
|---|---|---|
| やり方 | ありのまま観察 | 研究者が条件を**割り当てる** |
| 例 | コーヒーをよく飲む人を追跡 | くじで2群に分け、一方だけにコーヒー |
| 因果の主張 | しにくい（交絡が残る） | しやすい |

- **処理群（実験群）**：対策・薬などを与えるグループ
- **対照群（コントロール群）**：与えないで比べるグループ
- **無作為割り付け**：くじ引きで群を決める → 公平な比較に

> 観察研究で「相関」が見えても、原因は別（交絡因子）かもしれない。
原因を確かめたいときは、無作為化した**実験**が強い。

### 交絡（こうらく）の例
「アイスが売れる日は水難事故が多い」→ アイスが事故の原因ではなく、
両方の原因は**気温（夏）**。これが交絡因子です。実験ならランダム化で防げます。

## 2. 独立な試行

片方の結果がもう片方に影響しないとき**独立**といいます。
独立なら、両方が起こる確率はかけ算：
$$ P(A \cap B) = P(A) \times P(B) $$

例：コインが表(1/2) かつ サイコロが6(1/6)。

In [ ]:
p = (1/2) * (1/6)                          # 独立な事象は確率をかけ算
print('表 かつ 6 の確率:', round(p, 4))

## 3. 条件付き確率

「Bが起きたとき、Aが起きる確率」を $P(A \mid B)$ と書きます。
$$ P(A \mid B) = \frac{P(A \cap B)}{P(B)} $$

クロス集計表から計算するのが分かりやすいです。
例：ある学校の「部活の有無 × 朝食の有無」。

In [ ]:
# 「部活 × 朝食」のクロス集計表を作る
table = pd.DataFrame({'朝食あり':[180, 40], '朝食なし':[60, 120]},
                     index=['運動部', '帰宅部'])
table['合計'] = table.sum(axis=1)          # 行ごとの合計
print(table)

# 「運動部である」条件のもとで「朝食ありの確率」
p_break_given_sports = 180 / table.loc['運動部','合計']  # 条件付き確率 P(朝食あり|運動部)
print('P(朝食あり | 運動部) =', round(p_break_given_sports, 3))

# 全体での朝食ありの確率と比べる
p_break = (180+40) / table['合計'].sum()   # 全体の朝食あり率
print('P(朝食あり) =', round(p_break, 3), ' → 運動部の方が高い＝関連あり')

> $P(A\mid B) = P(A)$ のとき、AとBは独立（条件をつけても確率が変わらない）。

> **よくある間違い**：相関や関連が見えても、**交絡**（裏の共通原因）があるため因果は言えない。因果を確かめるには**無作為割付の実験**が強い。独立なら `P(A|B)=P(A)`。

## 4. ベイズの定理：陽性でも「本当に病気」とは限らない

めったにない病気（有病率が低い）では、検査で陽性でも実際に病気である確率は意外と低くなります。ベイズの定理と、10万人のシミュレーションの両方で確かめます。

In [ ]:
prev = 0.01     # 有病率（1%）
sens = 0.90     # 感度：病気の人が正しく陽性になる確率
spec = 0.90     # 特異度：健康な人が正しく陰性になる確率
# ベイズの定理：P(病気|陽性) = P(陽性|病気)P(病気) / P(陽性)
p_pos = sens*prev + (1-spec)*(1-prev)
ppv = sens*prev / p_pos
print(f'陽性的中率 P(病気|陽性) = {ppv:.3f}  → 陽性でも本当に病気は約{ppv:.0%}')
# 10万人で数えて確認
N = 100_000; sick = int(N*prev)
tp = sick*sens; fp = (N-sick)*(1-spec)
print(f'10万人中 陽性 {tp+fp:.0f}人 / うち実際に病気 {tp:.0f}人 → {tp/(tp+fp):.3f}')

> 有病率（**基準率**）が低いと、感度が高くても陽性的中率は低くなります。基準率を忘れて「陽性＝ほぼ病気」と思い込む誤りを **基準率の誤謬** といいます。
>
> スパム判定・不正検知・与信などBtoBの分類でも同じ：めったに起きない事象は、アラートが出ても「実際に該当」する割合が低くなりがちです。

## 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます（答えは表示されません）。

In [ ]:
# 採点用の関数を実行（答え合わせに使うだけ）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
# Q1: コインが表(1/2) かつ サイコロが6(1/6) の確率を ans に（独立）
ans = None   # (1/2)*(1/6)
_check('Q1 同時確率', ans, (1/2)*(1/6))

In [ ]:
# Q2: 運動部240人中180人が朝食あり。P(朝食あり|運動部) を ans に
ans = None   # 180/240
_check('Q2 条件付き確率', ans, 180/240)

In [ ]:
# Q3: 朝食なし180人中60人が運動部。P(運動部|朝食なし) を ans に
ans = None   # 60/180
_check('Q3 条件付き確率', ans, 60/180)

In [ ]:
# Q4: 有病率2%, 感度95%, 特異度90% のとき、陽性的中率 P(病気|陽性) を ans に
prev, sens, spec = 0.02, 0.95, 0.90
ans = None   # 例: sens*prev / (sens*prev + (1-spec)*(1-prev))
_check('Q4 陽性的中率', ans, sens*prev/(sens*prev+(1-spec)*(1-prev)))

---
## 練習問題

**問1.** 「睡眠アプリを使う人は成績が良い」という観察結果がある。
これだけで『アプリが成績を上げた』と言えない理由を、交絡の言葉で説明しよう。
また因果を確かめる実験のやり方を述べよう。

**問2.** サイコロを2回ふる。1回目が偶数 かつ 2回目が5 となる確率を、独立を使って求めよう。

**問3.** 上のクロス集計表で、`P(運動部 | 朝食なし)`（朝食なしの人が運動部である確率）を求めよう。

**問4.** 上の検査で、有病率だけを 1% → 10% に上げると陽性的中率はどう変わる？`prev` を変えて計算し、基準率が結果に効くことを確かめよう。

In [ ]:
# 問2


In [ ]:
# 問3


> **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから **[05_実験と条件付き確率 の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/03_%E7%B5%B1%E8%A8%88_3%E7%B4%9A/05_%E5%AE%9F%E9%A8%93%E3%81%A8%E6%9D%A1%E4%BB%B6%E4%BB%98%E3%81%8D%E7%A2%BA%E7%8E%87.md)**

## 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 観察研究 | ありのまま観察 |
| 実験研究 | 条件を割り当てて比較 |
| 処理群/対照群 | 対策あり/なしのグループ |
| 無作為割付 | くじ引きで群を決める |
| 交絡 | 両方に効く第3の原因 |
| 条件付き確率 | P(A\|B)=P(A∩B)/P(B) |
| 独立 | P(A∩B)=P(A)P(B) |

In [ ]:
# チートシート（確率）― そのまま実行できます
P_A, P_B = 1/2, 1/6
print('独立な同時確率 P(A∩B):', P_A * P_B)
import pandas as pd
t = pd.DataFrame({'あり':[180,40], 'なし':[60,120]}, index=['運動部','帰宅部'])
print('条件付き確率（行比率）:')
print((t.div(t.sum(axis=1), axis=0)).round(2))

## 完了ログ

このノートを終えたら下のセルを実行すると、学習の完了が記録されます。
**学習者コードは最初に一度だけ設定**すれば、以降は全ノートで自動送信されます（名前の再入力は不要）。

- Colab：左の鍵アイコン（シークレット）で `STUDENT_CODE` に配布コードを登録（1回だけ）
- ローカル：環境変数 `STUDENT_CODE` を設定（または初回に画面入力すると保存され、次回から不要）

In [ ]:
# === 完了ログ ===  学習者コードは最初に一度だけ設定すれば、全ノートで自動送信されます。
import os, datetime, pathlib
try:
    import requests
except ImportError:
    requests = None

def _student_code():
    try:                                          # Colab: 鍵アイコンで STUDENT_CODE を登録
        from google.colab import userdata
        c = userdata.get('STUDENT_CODE')
        if c: return c.strip()
    except Exception:
        pass
    c = os.environ.get('STUDENT_CODE')            # ローカル: 環境変数
    if c: return c.strip()
    p = pathlib.Path.home() / '.student_code'      # 保存ファイル
    if p.exists(): return p.read_text().strip()
    try:                                           # 最後の手段: 入力して保存（次回から不要）
        c = input('学習者コードを入力（配布されたもの）: ').strip()
        if c: p.write_text(c); return c
    except Exception:
        pass
    return ''

CODE = _student_code()
LOG_URL = ""      # 配布時に設定
LOG_TOKEN = ""    # 配布時に設定
NOTEBOOK = "03_統計_3級/05_実験と条件付き確率"

if CODE and LOG_URL and requests:
    try:
        requests.post(LOG_URL, json={"token": LOG_TOKEN, "code": CODE, "notebook": NOTEBOOK,
                      "time": datetime.datetime.now().isoformat(timespec="seconds")}, timeout=10)
        print(f"記録しました: {CODE} / {NOTEBOOK}")
    except Exception as e:
        print("記録に失敗しました（URL/ネットワークを確認）:", e)
elif not CODE:
    print("学習者コード未設定。Colabは鍵アイコンで STUDENT_CODE を登録、ローカルは環境変数 STUDENT_CODE を設定してください。")
else:
    print(f"{NOTEBOOK}: LOG_URL/LOG_TOKEN が未設定です（配布時に設定されます）。")